# MIMIC-IV W1 in-hospital mortality

This notebook reproduces the **W1** in-hospital mortality benchmark on MIMIC-IV, cell by cell. The cohort is the first ICU stay of each hospital admission; the label is death during the hospital stay (`hospital_expire_flag`). Vital and lab predictors are measured in the first 24 hours of the ICU stay, and the comorbidity feature is built only from the subject's prior, already-discharged admissions — so nothing observed after the prediction point can leak into the model.

It runs on the open-access 100-patient MIMIC-IV demo. Fetch it first with `python data/mimic_iv/fetch.py`. In-hospital mortality matters because it is the sharpest, least-ambiguous adverse outcome an early-warning model can target, and it anchors the discrimination-vs-calibration trade-off this benchmark measures.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Run from the benchmark directory so `import run` and `from src ...` resolve.
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from src import load_w1_cohort, fit_predict_proba, BetaBayesCalibrator
from src import evaluate_predictions, reliability_curve
from src.features import build_features_fit, build_features_apply, FEATURE_SPEC
from src.loader import LABEL_COL
from sklearn.model_selection import train_test_split
import run

SEED = 42
np.random.seed(SEED)

## 2. Load the cohort

Assemble the first-24h ICU-admission cohort from the MIMIC-IV tables and inspect its size, class balance, and missingness.

In [ ]:
cohort = load_w1_cohort()
frame = cohort.frame
print(f'admissions: {cohort.n}')
print(f'in-hospital mortality prevalence: {cohort.prevalence:.3f}')
print(f'deaths: {int(frame[LABEL_COL].sum())}')
frame.head()

In [ ]:
missing = frame.isna().mean().sort_values(ascending=False)
missing[missing > 0].round(3)

## 3. Features

The feature matrix has three blocks: one-hot admission descriptors, numeric demographics and the prior-history Charlson comorbidity index, and first-24h vital and lab means (median-imputed). Encoder and imputer are fit on training rows only; here we fit on the whole frame purely to inspect the column set.

In [ ]:
print('categorical:', FEATURE_SPEC['categorical'])
print('numeric:', FEATURE_SPEC['numeric'])
X_all, _ = build_features_fit(frame)
print('encoded columns:', X_all.shape[1])
X_all.iloc[:3]

## 4. One illustrative split

To show the mechanics we fit on a single stratified split. The headline numbers later come from the full cross-validated, multi-seed protocol; a single split on ~130 admissions is too noisy to trust on its own.

In [ ]:
y = frame[LABEL_COL].to_numpy(dtype=int)
feat = frame.drop(columns=[LABEL_COL])
tr, te = train_test_split(np.arange(len(y)), test_size=0.25,
                          random_state=SEED, stratify=y)
fit_idx, cal_idx = train_test_split(tr, test_size=0.3,
                                    random_state=SEED, stratify=y[tr])
X_fit, state = build_features_fit(feat.iloc[fit_idx])
X_cal = build_features_apply(feat.iloc[cal_idx], state)
X_te = build_features_apply(feat.iloc[te], state)
print('train/calib/test:', len(fit_idx), len(cal_idx), len(te))

## 5. Fit the model

LightGBM gradient-boosted trees, parameterised for a small imbalanced cohort.

In [ ]:
raw_cal = fit_predict_proba(X_fit, y[fit_idx], X_cal, random_state=SEED)
raw_te = fit_predict_proba(X_fit, y[fit_idx], X_te, random_state=SEED)
print('uncalibrated test AUROC:',
      round(evaluate_predictions(y[te], raw_te)['auroc'], 3))

## 6. Uncalibrated reliability

Boosted-trees probabilities are typically over-confident. The reliability curve plots observed positive rate against predicted probability; a perfectly calibrated model sits on the diagonal.

In [ ]:
def plot_reliability(y_true, prob, title):
    rc = reliability_curve(y_true, prob, n_bins=8)
    plt.figure(figsize=(4, 4))
    plt.plot([0, 1], [0, 1], '--', color='grey', label='ideal')
    plt.plot(rc['mean_pred'], rc['frac_pos'], 'o-', label='model')
    plt.xlabel('mean predicted probability')
    plt.ylabel('observed positive rate')
    plt.title(title)
    plt.legend(); plt.tight_layout(); plt.show()

plot_reliability(y[te], raw_te, 'Uncalibrated')

## 7. Beta + Bayes calibration

Fit the Beta calibrator on the held-out calibration slice, then apply it to the test scores and re-plot.

In [ ]:
cal = BetaBayesCalibrator().fit(raw_cal, y[cal_idx])
cal_te = cal.predict(raw_te)
print('uncalibrated ECE:', round(evaluate_predictions(y[te], raw_te)['ece'], 4))
print('calibrated   ECE:', round(evaluate_predictions(y[te], cal_te)['ece'], 4))
plot_reliability(y[te], cal_te, 'Beta + Bayes calibrated')

## 8. Headline numbers (cross-validated, multi-seed)

Now the stable protocol: pooled out-of-fold predictions from 5-fold cross-validation, averaged over several seeds. (The canonical contract uses 40 seeds; a few are enough to see the numbers here.)

In [ ]:
results = run.run_reproducer(None, seed=SEED, n_seeds=5, folds=5)
import pandas as pd
pd.DataFrame({'uncalibrated': results['uncalibrated'],
              'calibrated': results['calibrated']}).round(4)

## 9. Compare to the expected contract

The committed `expected_results.json` pins the canonical 40-seed demo numbers and their tolerances. Small seed counts here will sit close to, but not exactly on, those targets.

In [ ]:
import json
expected = json.loads(Path('expected_results.json').read_text())
demo = expected['reproducer_targets']['demo']
for k in ['auroc', 'average_precision', 'brier', 'ece']:
    obs = results['calibrated'][k]
    print(f"{k:20s} observed {obs:.4f}  target {demo[k]['target']:.4f} "
          f"+/- {demo[k]['tol']}")

## 10. Interpretation

On the 100-patient demo, discrimination is modest and noisy: ~130 admissions with ~15 deaths simply cannot support a high, stable AUROC, which is why the reproducer cross-validates and averages rather than trusting one split. What the demo does show cleanly is that the Beta + Bayes calibrator tightens calibration — it roughly halves the uncalibrated ECE — while leaving a working, fully deterministic, public pipeline that anyone can run.

The full-corpus figure is a different regime. On the credentialed 546K-admission corpus the same family of methods reaches the AUROC recorded in `expected_results.json` under `internal_reference`; that number is not reproducible here because the corpus is not (and cannot be) committed.

## 11. Internal reference

The full internal system augments this pipeline with additional risk-stratification and calibration components; the figures in `expected_results.json` -> `internal_reference` are from that system on the full corpus. Those extensions are commercial. The match between the public methods here and the corresponding claim demonstrates the result is real without exposing how the closed platform achieves it.